# Smart Support Desk

##### Imagine an e-commerce store gets hundreds of support emails. Instead of humans reading every single email, your local system reads the email, searches ChromaDB for standard fixes, categorizes the priority (High/Low), and drafts a response using SLM/LLM.

In [69]:
from openai import OpenAI
import chromadb
import requests
import os
from dotenv import load_dotenv
load_dotenv()

# Function i made to call LLM
def chat_minicpm(question):
    client = OpenAI(
        base_url="https://api.modelbest.cn/v1",
        api_key=os.environ['MINICPM_KEY']
    )

    response = client.chat.completions.create(
        model="MiniCPM-V-4.6-Thinking",
        messages=[
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content


# Querying the chroma db with the user question, and creating context for LLM
def process_incoming_ticket(email_text):
    search_results = collection.query(query_texts=[email_text], n_results=2)
    matched_policy = search_results['documents'][0][0] 
    
    context = f"""You are an experienced customer support representative. 
Analyze the customer email using the provided Company Policy.
keep close eye to detail and reply thinking twice about the question and its details.

Company Policy:
{matched_policy}

Customer Email:
"{email_text}"

Your output must follow this exact template:
PRIORITY: [Urgent / Low]
CATEGORY: [Refund / Shipping / Technical]
DRAFT REPLY: [Write a polite reply based ONLY on the company policy. If policy doesn't match, say a human agent will review it shortly.]
"""

    return chat_minicpm(context)

In [ ]:
# testing the LLM response
print(chat_minicpm("Hello MiniCPM!!"))



Hello there! 🎉 I'm excited to see you reach out to me!

I'm a model from the MiniCPM series, developed by ModelBest and the OpenBMB community. It's great to meet your attention!

As part of this series, I'm known for being efficient and powerful, designed to deliver strong performance with smaller parameter counts. Whether you need help with coding, answering questions, or just want to chat, I'm here to assist.

What can I do for you today? 😊


In [ ]:
# Sample Data on policies

company_knowledge = {
    "ids": [
        "policy_1",
        "policy_2",
        "policy_3",
        "policy_4",
        "policy_5",
        "policy_6",
    ],
    "chunks": [
        "Refund Policy: Customers can get a full refund within 30 days of purchase if the item is unused.",
        "Shipping Delay: standard delivery takes 3-5 business days. Express shipping takes 1-2 days.",
        "Technical Crash: If the app crashes, instruct user to clear cache, restart app, or re-install.",
        "Exchange Policy: Items can be exchanged for a different size or color within 45 days of purchase, even if opened, as long as they are undamaged.",
        "Damaged Items: If an item arrives damaged, the customer must provide a photo within 48 hours of delivery to receive a free replacement.",
        "Payment Methods: We accept all major credit cards, PayPal, and Apple Pay. We do not accept cash on delivery or personal checks.",
    ],
}


In [ ]:
# Initialize and populate local database
client = chromadb.PersistentClient(path="./company_support_db")
collection = client.get_or_create_collection("faq_collection")
collection.upsert(documents=company_knowledge['chunks'], ids=company_knowledge['ids'])


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [49]:
questions = ["Can I get a refund if I bought a jacket 10 days ago but wore it once?",
"My order hasn't arrived yet. It has been 4 business days since I chose standard shipping. Is it delayed?",
"The app keeps closing automatically when I click checkout. What should I do?",
"I bought a dress 40 days ago and never opened it. Can I return it for cash?",
"I paid for express shipping yesterday morning. When should my package arrive?",
"The application is completely frozen on the login screen. How do I fix this?",
"I bought a t-shirt 3 weeks ago, tried it on, and it doesn't fit. It is completely unused. Can I get a full refund?",
"My standard delivery package is arriving on the 6th business day. Can I get a shipping refund?",
"I uninstalled and reinstalled the app, but it still crashes on launch. What is the next step?",
"Can I return an item after 25 days if I threw away the original packaging but never used the item?"
]

In [70]:
for i in questions:
    print("\n------------------- Email ----------------------")
    print("Email: ",i)
    print("----------------- Response ----------------------")
    print("Response: ",process_incoming_ticket(i))
    print("----------------x Response x---------------------")
    print()



------------------- Email ----------------------
Email:  Can I get a refund if I bought a jacket 10 days ago but wore it once?
----------------- Response ----------------------
Response:  

PRIORITY: Low  
CATEGORY: Refund  

DRAFT REPLY: Thank you for contacting us regarding your refund request. Based on our Refund Policy, you are eligible for a full refund within 30 days of purchase if the item is unused. Since you bought the jacket 10 days ago and it's still within the 30-day window, if the jacket remains in new condition and not worn at all, you should be eligible for a full refund. However, if you wore it once, it may no longer be considered "unused," which could impact the refund eligibility. I recommend verifying the jacket's condition and confirming it meets the policy requirements before proceeding.
----------------x Response x---------------------


------------------- Email ----------------------
Email:  My order hasn't arrived yet. It has been 4 business days since I chose